In [1]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, StratifiedShuffleSplit,
                                     StratifiedGroupKFold)
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, matthews_corrcoef,
                              confusion_matrix)
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel
from qiskit.circuit.library import zz_feature_map, z_feature_map
from qiskit.quantum_info import Statevector
import qiskit

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Qiskit version:", qiskit.__version__)
print("Random seed:", RANDOM_SEED)

Qiskit version: 1.4.5
Random seed: 42


In [2]:
# --- Celda 1: cargar features (nuevo) + etiquetas (del Excel original, como antes) ---
data = pd.read_csv("PC1_data_frame.csv")

# por si el csv trae una columna de id/índice tipo "Unnamed: 0" o "row" -- se descarta,
# solo nos quedamos con las columnas numéricas de features
non_feature_cols = [c for c in data.columns if not pd.api.types.is_numeric_dtype(data[c])]
if non_feature_cols:
    print("Columnas no-numéricas descartadas de X:", non_feature_cols)
X = data.drop(columns=non_feature_cols).values

_seq_df = pd.read_excel("Docking_high_low_energy_labels.xlsx", sheet_name="Results")
_seq_df = _seq_df.dropna(subset=["Sequence Epitope)"]).reset_index(drop=True)
sequences = _seq_df["Sequence Epitope)"].astype(str).str.strip().tolist()
y = (_seq_df["Otsu Class theshold -77.8"] == "Strong").astype(int).values

assert len(sequences) == len(y) == X.shape[0], \
    "el numero de filas de z_scaled_5_45.csv debe coincidir con las filas del Excel de etiquetas"

print("Shape:", X.shape, " Class balance:", np.bincount(y))

Shape: (80, 9)  Class balance: [42 38]


/home/luispabloelchido/miniconda3/envs/QML/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [3]:
# --- Celda 2: agrupamiento por redundancia de secuencia (igual que antes, sin cambios) ---
K_SHARED = 5
IDENTITY_THRESH = 7/9

def _kmers(s, k=K_SHARED):
    return {s[i:i+k] for i in range(len(s) - k + 1)}

_n = len(sequences)
_parent = list(range(_n))
def _find(a):
    while _parent[a] != a:
        _parent[a] = _parent[_parent[a]]
        a = _parent[a]
    return a
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        _parent[ra] = rb

_kmer_sets = [_kmers(s) for s in sequences]
for i in range(_n):
    for j in range(i + 1, _n):
        if _kmer_sets[i] & _kmer_sets[j]:
            _union(i, j)
            continue
        identity = sum(a == b for a, b in zip(sequences[i], sequences[j])) / 9
        if identity >= IDENTITY_THRESH:
            _union(i, j)

_raw_groups = [_find(i) for i in range(_n)]
_remap = {g: idx for idx, g in enumerate(sorted(set(_raw_groups)))}
groups = np.array([_remap[g] for g in _raw_groups])

In [4]:
def fast_statevector_kernel(X_a, X_b, feature_map):
    """Exact fidelity kernel via statevector overlap -- see markdown note on the
    Loschmidt-echo circuit above. Verified against FidelityQuantumKernel.evaluate
    to ~1e-12, ~500-600x faster."""
    def statevectors(X):
        return np.array([Statevector.from_instruction(feature_map.assign_parameters(x)).data for x in X])
    sv_a = statevectors(X_a)
    sv_b = sv_a if X_b is None else statevectors(X_b)
    return np.abs(sv_a.conj() @ sv_b.T) ** 2


def build_feature_map(kind, n_features, reps, entanglement='linear'):
    if kind == 'zz':
        return zz_feature_map(n_features, reps=reps, entanglement=entanglement)
    elif kind == 'z':
        return z_feature_map(n_features, reps=reps)
    raise ValueError(kind)


def fit_scale(X_train, X_test, bandwidth=1.0):
    """Fix 2.2: a NEW scaler per fold, fit ONLY on training data."""
    scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
    scaler.fit(X_train)
    return scaler.transform(X_train) * bandwidth, scaler.transform(X_test) * bandwidth


def kernel_diagnostics(K, y_pm1):
    """Returns (KTA, CKA, effective_rank) for a train-fold Gram matrix K
    and labels y_pm1 in {-1, +1}, using Steve's corrected formulas."""
    m = len(y_pm1)
    S = y_pm1 @ K @ y_pm1 - np.trace(K)
    Q = np.sum(K ** 2) - np.sum(np.diag(K) ** 2)
    kta = (m + S) / (m * np.sqrt(m + Q))

    H = np.eye(m) - np.ones((m, m)) / m
    Kc = H @ K @ H
    yyT = np.outer(y_pm1, y_pm1).astype(float)
    yyTc = H @ yyT @ H
    cka = np.sum(Kc * yyTc) / (np.linalg.norm(Kc) * np.linalg.norm(yyTc) + 1e-15)

    eigvals = np.clip(np.linalg.eigvalsh((K + K.T) / 2), 0, None)
    eff_rank = (eigvals.sum() ** 2) / (np.sum(eigvals ** 2) + 1e-15)
    return kta, cka, eff_rank


def check_kernel_psd(K, tol=-1e-8):
    eigvals = np.linalg.eigvalsh((K + K.T) / 2)
    return eigvals.min() >= tol, eigvals.min()

In [5]:
def nested_cv_quantum(X, y, param_grid, kind, n_splits_outer=5, n_repeats_outer=20,
                      n_splits_inner=4, entanglement='linear',
                      random_state=RANDOM_SEED, verbose=False, groups=None):
    if groups is None:
        outer_splits = list(RepeatedStratifiedKFold(n_splits=n_splits_outer, n_repeats=n_repeats_outer,
                                                      random_state=random_state).split(X, y))
    else:
        outer_splits = repeated_stratified_group_kfold(y, groups, n_splits=n_splits_outer,
                                                         n_repeats=n_repeats_outer, random_state=random_state)
    rows = []
    predictions = {}
    t0 = time.time()
    for fold_i, (tr_idx, te_idx) in enumerate(outer_splits):
        X_tr_outer, X_te_outer = X[tr_idx], X[te_idx]
        y_tr_outer, y_te_outer = y[tr_idx], y[te_idx]

        if groups is None:
            inner_splits = list(StratifiedKFold(n_splits=n_splits_inner, shuffle=True,
                                                 random_state=random_state + fold_i).split(X_tr_outer, y_tr_outer))
        else:
            groups_tr_outer = groups[tr_idx]
            inner_splits = list(StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True,
                                                       random_state=random_state + fold_i)
                                 .split(X_tr_outer, y_tr_outer, groups_tr_outer))
        inner_scores = {}
        for reps in param_grid['reps']:
            for bw in param_grid['bandwidth']:
                for in_tr_idx, in_val_idx in inner_splits:
                    Xi_tr, Xi_val = X_tr_outer[in_tr_idx], X_tr_outer[in_val_idx]
                    yi_tr, yi_val = y_tr_outer[in_tr_idx], y_tr_outer[in_val_idx]
                    Xi_tr_s, Xi_val_s = fit_scale(Xi_tr, Xi_val, bandwidth=bw)
                    fmap = build_feature_map(kind, X.shape[1], reps, entanglement)
                    K_tr = fast_statevector_kernel(Xi_tr_s, None, fmap)
                    K_val = fast_statevector_kernel(Xi_val_s, Xi_tr_s, fmap)
                    for C in param_grid['C']:
                        clf = SVC(kernel='precomputed', C=C)
                        clf.fit(K_tr, yi_tr)
                        score = balanced_accuracy_score(yi_val, clf.predict(K_val))
                        inner_scores.setdefault((reps, bw, C), []).append(score)

        best_reps, best_bw, best_C = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))

        Xtr_s, Xte_s = fit_scale(X_tr_outer, X_te_outer, bandwidth=best_bw)
        fmap = build_feature_map(kind, X.shape[1], best_reps, entanglement)
        K_tr = fast_statevector_kernel(Xtr_s, None, fmap)
        K_te = fast_statevector_kernel(Xte_s, Xtr_s, fmap)
        clf = SVC(kernel='precomputed', C=best_C)
        clf.fit(K_tr, y_tr_outer)
        y_pred = clf.predict(K_te)

        y_tr_pm1 = np.where(y_tr_outer == 1, 1, -1)
        kta, cka, eff_rank = kernel_diagnostics(K_tr, y_tr_pm1)

        rows.append({
            'fold': fold_i, 'best_reps': best_reps, 'best_bandwidth': best_bw, 'best_C': best_C,
            'accuracy': accuracy_score(y_te_outer, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_te_outer, y_pred),
            'mcc': matthews_corrcoef(y_te_outer, y_pred) if len(set(y_pred)) > 1 else 0.0,
            'kta': kta, 'cka': cka, 'effective_rank': eff_rank,
            'n_test': len(te_idx),
        })
        predictions[fold_i] = {'y_true': y_te_outer.tolist(), 'y_pred': y_pred.tolist()}
        if verbose and fold_i % 10 == 0:
            print(f'[{kind}] fold {fold_i} done, {time.time()-t0:.1f}s')
    return pd.DataFrame(rows), predictions

In [6]:
CLASSICAL_GRIDS = {
    'linear': {'C': [0.1, 1, 10]},
    'rbf':    {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1, 1]},
    'poly':   {'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 0.1, 1], 'coef0': [0.0, 1.0]},
}

def _resolve_gamma(gamma, Xs):
    if gamma == 'scale':
        return 1.0 / (Xs.shape[1] * Xs.var()) if Xs.var() > 0 else 1.0
    return gamma

def _build_gram(kernel_name, Xa, Xb, params, Xtr_for_scale):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr_for_scale)
    if kernel_name == 'linear':
        return linear_kernel(Xa, Xb)
    elif kernel_name == 'rbf':
        return rbf_kernel(Xa, Xb, gamma=gamma)
    elif kernel_name == 'poly':
        return polynomial_kernel(Xa, Xb, degree=params['degree'], gamma=gamma, coef0=params['coef0'])
    raise ValueError(kernel_name)

def _fit_svc(kernel_name, Xtr, ytr, params, Xtr_for_scale):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr_for_scale)
    if kernel_name == 'linear':
        clf = SVC(kernel='linear', C=params['C'])
    elif kernel_name == 'rbf':
        clf = SVC(kernel='rbf', C=params['C'], gamma=gamma)
    elif kernel_name == 'poly':
        clf = SVC(kernel='poly', C=params['C'], degree=params['degree'], gamma=gamma, coef0=params['coef0'])
    clf.fit(Xtr, ytr)
    return clf

def _param_combinations(grid):
    keys = list(grid.keys())
    def rec(i, acc):
        if i == len(keys):
            yield dict(acc)
            return
        for v in grid[keys[i]]:
            acc[keys[i]] = v
            yield from rec(i + 1, acc)
    return list(rec(0, {}))


def nested_cv_classical(X, y, kernel_name, n_splits_outer=5, n_repeats_outer=20,
                        n_splits_inner=4, random_state=RANDOM_SEED, verbose=False, groups=None):
    grid = CLASSICAL_GRIDS[kernel_name]
    combos = _param_combinations(grid)
    if groups is None:
        outer_splits = list(RepeatedStratifiedKFold(n_splits=n_splits_outer, n_repeats=n_repeats_outer,
                                                      random_state=random_state).split(X, y))
    else:
        outer_splits = repeated_stratified_group_kfold(y, groups, n_splits=n_splits_outer,
                                                         n_repeats=n_repeats_outer, random_state=random_state)
    rows = []
    predictions = {}
    t0 = time.time()
    for fold_i, (tr_idx, te_idx) in enumerate(outer_splits):
        X_tr_outer, X_te_outer = X[tr_idx], X[te_idx]
        y_tr_outer, y_te_outer = y[tr_idx], y[te_idx]

        if groups is None:
            inner_splits = list(StratifiedKFold(n_splits=n_splits_inner, shuffle=True,
                                                 random_state=random_state + fold_i).split(X_tr_outer, y_tr_outer))
        else:
            groups_tr_outer = groups[tr_idx]
            inner_splits = list(StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True,
                                                       random_state=random_state + fold_i)
                                 .split(X_tr_outer, y_tr_outer, groups_tr_outer))
        inner_scores = {i: [] for i in range(len(combos))}
        for in_tr_idx, in_val_idx in inner_splits:
            Xi_tr, Xi_val = X_tr_outer[in_tr_idx], X_tr_outer[in_val_idx]
            yi_tr, yi_val = y_tr_outer[in_tr_idx], y_tr_outer[in_val_idx]
            Xi_tr_s, Xi_val_s = fit_scale(Xi_tr, Xi_val)
            for ci, params in enumerate(combos):
                clf = _fit_svc(kernel_name, Xi_tr_s, yi_tr, params, Xi_tr_s)
                score = balanced_accuracy_score(yi_val, clf.predict(Xi_val_s))
                inner_scores[ci].append(score)

        best_ci = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))
        best_params = combos[best_ci]

        Xtr_s, Xte_s = fit_scale(X_tr_outer, X_te_outer)
        clf = _fit_svc(kernel_name, Xtr_s, y_tr_outer, best_params, Xtr_s)
        y_pred = clf.predict(Xte_s)

        K_tr = _build_gram(kernel_name, Xtr_s, Xtr_s, best_params, Xtr_s)
        y_tr_pm1 = np.where(y_tr_outer == 1, 1, -1)
        kta, cka, eff_rank = kernel_diagnostics(K_tr, y_tr_pm1)

        row = {'fold': fold_i,
               'accuracy': accuracy_score(y_te_outer, y_pred),
               'balanced_accuracy': balanced_accuracy_score(y_te_outer, y_pred),
               'mcc': matthews_corrcoef(y_te_outer, y_pred) if len(set(y_pred)) > 1 else 0.0,
               'kta': kta, 'cka': cka, 'effective_rank': eff_rank,
               'n_test': len(te_idx)}
        row.update({f'best_{k}': v for k, v in best_params.items()})
        rows.append(row)
        predictions[fold_i] = {'y_true': y_te_outer.tolist(), 'y_pred': y_pred.tolist()}
        if verbose and fold_i % 20 == 0:
            print(f'[classical-{kernel_name}] fold {fold_i} done, {time.time()-t0:.1f}s')
    return pd.DataFrame(rows), predictions


def majority_baseline(X, y, n_splits_outer=5, n_repeats_outer=20, random_state=RANDOM_SEED, groups=None):
    if groups is None:
        outer_splits = list(RepeatedStratifiedKFold(n_splits=n_splits_outer, n_repeats=n_repeats_outer,
                                                      random_state=random_state).split(X, y))
    else:
        outer_splits = repeated_stratified_group_kfold(y, groups, n_splits=n_splits_outer,
                                                         n_repeats=n_repeats_outer, random_state=random_state)
    rows, predictions = [], {}
    for fold_i, (tr_idx, te_idx) in enumerate(outer_splits):
        clf = DummyClassifier(strategy='most_frequent')
        clf.fit(X[tr_idx], y[tr_idx])
        y_pred = clf.predict(X[te_idx])
        rows.append({'fold': fold_i, 'accuracy': accuracy_score(y[te_idx], y_pred),
                     'balanced_accuracy': balanced_accuracy_score(y[te_idx], y_pred),
                     'mcc': matthews_corrcoef(y[te_idx], y_pred) if len(set(y_pred)) > 1 else 0.0})
        predictions[fold_i] = {'y_true': y[te_idx].tolist(), 'y_pred': y_pred.tolist()}
    return pd.DataFrame(rows), predictions

In [7]:
def make_ratio_outer_cv(test_size, n_repeats=20, random_state=RANDOM_SEED):
    """e.g. make_ratio_outer_cv(0.3) for a 70/30 split, repeated 20 times.
    Use in place of RepeatedStratifiedKFold(...).split(X, y): both yield
    (train_idx, test_idx) pairs, so nested_cv_quantum / nested_cv_classical
    do not need to change -- only the outer_cv object construction inside them."""
    return StratifiedShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=random_state)

# Example (not executed): to sweep 90/10 down to 10/90 skipping 40/60 and 60/40,
# as in the original manuscript, you would loop:
# for test_size in [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]:
#     outer_cv = make_ratio_outer_cv(test_size)
#     ... same nested-CV body, using outer_cv.split(X, y) instead of RepeatedStratifiedKFold(...).split(X, y) ...

In [ ]:
PARAM_GRID_QUANTUM = {'reps': [1, 2], 'C': [0.1, 1, 10], 'bandwidth': [0.05, 0.1, 0.2 , 0.5]}
N_SPLITS_OUTER, N_REPEATS_OUTER, N_SPLITS_INNER = 5, 20, 4

t0 = time.time()
res_zz, preds_zz = nested_cv_quantum(X, y, PARAM_GRID_QUANTUM, kind='zz',
                                      n_splits_outer=N_SPLITS_OUTER, n_repeats_outer=N_REPEATS_OUTER,
                                      n_splits_inner=N_SPLITS_INNER, verbose=True)
print(f'ZZFeatureMap: {time.time()-t0:.1f}s total')
res_zz.to_csv('results_quantum_zz.csv', index=False)
json.dump(preds_zz, open('preds_zz.json', 'w'))

[zz] fold 0 done, 15.8s
[zz] fold 10 done, 158.6s
[zz] fold 20 done, 299.4s
[zz] fold 30 done, 442.2s


In [ ]:
t0 = time.time()
res_z, preds_z = nested_cv_quantum(X, y, PARAM_GRID_QUANTUM, kind='z',
                                    n_splits_outer=N_SPLITS_OUTER, n_repeats_outer=N_REPEATS_OUTER,
                                    n_splits_inner=N_SPLITS_INNER, verbose=True)
print(f'ZFeatureMap: {time.time()-t0:.1f}s total')
res_z.to_csv('results_quantum_z.csv', index=False)
json.dump(preds_z, open('preds_z.json', 'w'))

[z] fold 0 done, 7.0s
[z] fold 10 done, 73.1s
[z] fold 20 done, 139.9s
[z] fold 30 done, 207.2s
[z] fold 40 done, 276.9s
[z] fold 50 done, 344.1s
[z] fold 60 done, 411.4s
[z] fold 70 done, 478.8s
[z] fold 80 done, 547.6s
[z] fold 90 done, 615.4s
ZFeatureMap: 675.4s total


In [ ]:
classical_results, classical_preds = {}, {}
for kname in ['linear', 'rbf', 'poly']:
    t0 = time.time()
    res, preds = nested_cv_classical(X, y, kname, n_splits_outer=N_SPLITS_OUTER,
                                      n_repeats_outer=N_REPEATS_OUTER, n_splits_inner=N_SPLITS_INNER,
                                      verbose=True)
    print(f'Classical {kname}: {time.time()-t0:.1f}s total')
    res.to_csv(f'results_classical_{kname}.csv', index=False)
    json.dump(preds, open(f'preds_classical_{kname}.json', 'w'))
    classical_results[kname] = res
    classical_preds[kname] = preds

res_majority, preds_majority = majority_baseline(X, y, n_splits_outer=N_SPLITS_OUTER,
                                                  n_repeats_outer=N_REPEATS_OUTER)
res_majority.to_csv('results_majority.csv', index=False)
json.dump(preds_majority, open('preds_majority.json', 'w'))

[classical-linear] fold 0 done, 0.1s
[classical-linear] fold 20 done, 1.3s
[classical-linear] fold 40 done, 2.4s
[classical-linear] fold 60 done, 3.5s
[classical-linear] fold 80 done, 4.5s
Classical linear: 5.5s total
[classical-rbf] fold 0 done, 0.1s
[classical-rbf] fold 20 done, 1.7s
[classical-rbf] fold 40 done, 3.4s
[classical-rbf] fold 60 done, 5.0s
[classical-rbf] fold 80 done, 6.6s
Classical rbf: 8.1s total
[classical-poly] fold 0 done, 0.3s
[classical-poly] fold 20 done, 4.9s
[classical-poly] fold 40 done, 9.4s
[classical-poly] fold 60 done, 13.9s
[classical-poly] fold 80 done, 18.4s
Classical poly: 22.8s total


In [ ]:
def per_fold_metrics(predictions):
    accs, baccs, mccs, sens, specs = [], [], [], [], []
    y_true_all, y_pred_all = [], []
    for fold_i in sorted(predictions):
        yt = np.array(predictions[fold_i]['y_true'])
        yp = np.array(predictions[fold_i]['y_pred'])
        y_true_all.extend(yt.tolist()); y_pred_all.extend(yp.tolist())
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
        accs.append(accuracy_score(yt, yp))
        baccs.append(balanced_accuracy_score(yt, yp))
        mccs.append(matthews_corrcoef(yt, yp) if len(set(yp)) > 1 else 0.0)
    agg_cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
    return {'accuracy': np.array(accs), 'balanced_accuracy': np.array(baccs), 'mcc': np.array(mccs),
            'sensitivity': np.array(sens), 'specificity': np.array(specs),
            'aggregate_confusion_matrix': agg_cm}

metrics = {
    'ZZFeatureMap (entangled)': per_fold_metrics(preds_zz),
    'ZFeatureMap (no entanglement)': per_fold_metrics(preds_z),
    'Classical - linear': per_fold_metrics(classical_preds['linear']),
    'Classical - RBF': per_fold_metrics(classical_preds['rbf']),
    'Classical - poly': per_fold_metrics(classical_preds['poly']),
    'Majority-class baseline': per_fold_metrics(preds_majority),
}

for name, m in metrics.items():
    print(f'--- {name} ---')
    print(m['aggregate_confusion_matrix'])
    print(f"sens={np.nanmean(m['sensitivity']):.4f}  spec={np.nanmean(m['specificity']):.4f}")
    print()

    # a traves de todas las folds (1600) total

--- ZZFeatureMap (entangled) ---
[[595 245]
 [451 309]]
sens=0.4079  spec=0.7097

--- ZFeatureMap (no entanglement) ---
[[513 327]
 [383 377]]
sens=0.4995  spec=0.6110

--- Classical - linear ---
[[445 395]
 [419 341]]
sens=0.4491  spec=0.5296

--- Classical - RBF ---
[[550 290]
 [413 347]]
sens=0.4618  spec=0.6572

--- Classical - poly ---
[[550 290]
 [386 374]]
sens=0.4932  spec=0.6550

--- Majority-class baseline ---
[[840   0]
 [760   0]]
sens=0.0000  spec=1.0000



## Bootstrap paired confidence intervals

In [ ]:
def bootstrap_paired_ci(diff, n_boot=10000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n = len(diff)
    boots = np.array([rng.choice(diff, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff.mean(), lo, hi

comparisons = {
    'ZZ - Z': metrics['ZZFeatureMap (entangled)']['balanced_accuracy'] - metrics['ZFeatureMap (no entanglement)']['balanced_accuracy'],
    'ZZ - Classical RBF': metrics['ZZFeatureMap (entangled)']['balanced_accuracy'] - metrics['Classical - RBF']['balanced_accuracy'],
    'ZZ - Classical linear': metrics['ZZFeatureMap (entangled)']['balanced_accuracy'] - metrics['Classical - linear']['balanced_accuracy'],
    'ZZ - Classical poly': metrics['ZZFeatureMap (entangled)']['balanced_accuracy'] - metrics['Classical - poly']['balanced_accuracy'],
    'Z - Classical RBF': metrics['ZFeatureMap (no entanglement)']['balanced_accuracy'] - metrics['Classical - RBF']['balanced_accuracy'],
}
for name, diff in comparisons.items():
    mean, lo, hi = bootstrap_paired_ci(diff)
    print(f'{name}: mean={mean:+.4f}  95% CI=[{lo:+.4f}, {hi:+.4f}]  '
          f'{"(excludes zero)" if (lo>0 or hi<0) else "(includes zero)"}')

ZZ - Z: mean=+0.0036  95% CI=[-0.0173, +0.0252]  (includes zero)
ZZ - Classical RBF: mean=-0.0007  95% CI=[-0.0231, +0.0213]  (includes zero)
ZZ - Classical linear: mean=+0.0694  95% CI=[+0.0469, +0.0926]  (excludes zero)
ZZ - Classical poly: mean=-0.0153  95% CI=[-0.0385, +0.0080]  (includes zero)
Z - Classical RBF: mean=-0.0043  95% CI=[-0.0235, +0.0138]  (includes zero)


## Final summary — all metrics in one table

In [ ]:
summary_rows = []
for name, m in metrics.items():
    row = {'model': name, 'n_outer_folds': len(m['accuracy'])}
    for key in ['accuracy', 'balanced_accuracy', 'mcc', 'sensitivity', 'specificity']:
        vals = m[key]
        row[f'{key}_mean'] = np.nanmean(vals)
        row[f'{key}_se'] = np.nanstd(vals, ddof=1) / np.sqrt(np.sum(~np.isnan(vals)))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index('model')

kernel_diag_sources = {
    'ZZFeatureMap (entangled)': res_zz, 'ZFeatureMap (no entanglement)': res_z,
    'Classical - linear': classical_results['linear'], 'Classical - RBF': classical_results['rbf'],
    'Classical - poly': classical_results['poly'],
}
diag_rows = []
for name, res in kernel_diag_sources.items():
    diag_rows.append({
        'model': name,
        'kta_mean': res['kta'].mean(), 'kta_se': res['kta'].std(ddof=1) / np.sqrt(len(res)),
        'cka_mean': res['cka'].mean(), 'cka_se': res['cka'].std(ddof=1) / np.sqrt(len(res)),
        'effective_rank_mean': res['effective_rank'].mean(),
        'effective_rank_se': res['effective_rank'].std(ddof=1) / np.sqrt(len(res)),
    })
diag_df = pd.DataFrame(diag_rows).set_index('model')

final_summary = summary_df.join(diag_df, how='left')
final_summary.to_csv('final_summary_metrics.csv')

comp_rows = []
for name, diff in comparisons.items():
    mean, lo, hi = bootstrap_paired_ci(diff)
    comp_rows.append({'comparison': name, 'mean_diff': mean, 'ci_low': lo, 'ci_high': hi,
                       'excludes_zero': (lo > 0) or (hi < 0)})
comp_df = pd.DataFrame(comp_rows).set_index('comparison')
comp_df.to_csv('final_summary_comparisons.csv')

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print("=" * 110)
print("FINAL SUMMARY -- classification metrics, mean +/- SE over 100 paired outer test folds")
print("=" * 110)
display(final_summary[['accuracy_mean','accuracy_se','balanced_accuracy_mean','balanced_accuracy_se',
                        'mcc_mean','mcc_se','sensitivity_mean','specificity_mean']])

print()
print("=" * 110)
print("KERNEL DIAGNOSTICS -- KTA / CKA / effective rank, mean +/- SE (classical models: no KTA/CKA if not applicable)")
print("=" * 110)
display(final_summary[['kta_mean','kta_se','cka_mean','cka_se','effective_rank_mean','effective_rank_se']])

print()
print("Pairwise bootstrap comparisons (balanced accuracy):")
display(comp_df)

print()
print("Most frequently selected quantum hyperparameters:")
print("ZZ - bandwidth:", res_zz['best_bandwidth'].value_counts().to_dict(),
      " reps:", res_zz['best_reps'].value_counts().to_dict())
print("Z  - bandwidth:", res_z['best_bandwidth'].value_counts().to_dict(),
      " reps:", res_z['best_reps'].value_counts().to_dict())

FINAL SUMMARY -- classification metrics, mean +/- SE over 100 paired outer test folds


,accuracy_mean,accuracy_se,balanced_accuracy_mean,balanced_accuracy_se,mcc_mean,mcc_se,sensitivity_mean,specificity_mean
model,,,,,,,,
ZZFeatureMap (entangled),0.5650,0.0094,0.5588,0.0094,0.1339,0.0212,0.4079,0.7097
ZFeatureMap (no entanglement),0.5563,0.0106,0.5552,0.0107,0.1222,0.0231,0.4995,0.6110
Classical - linear,0.4913,0.0100,0.4893,0.0100,-0.0231,0.0205,0.4491,0.5296
Classical - RBF,0.5606,0.0099,0.5595,0.0101,0.1294,0.0216,0.4618,0.6572
Classical - poly,0.5775,0.0110,0.5741,0.0109,0.1573,0.0230,0.4932,0.6550
Majority-class baseline,0.5250,0.0031,0.5000,0.0000,0.0000,0.0000,0.0000,1.0000



KERNEL DIAGNOSTICS -- KTA / CKA / effective rank, mean +/- SE (classical models: no KTA/CKA if not applicable)


,kta_mean,kta_se,cka_mean,cka_se,effective_rank_mean,effective_rank_se
model,,,,,,
ZZFeatureMap (entangled),0.1114,0.0020,0.1171,0.0016,44.4479,1.3386
ZFeatureMap (no entanglement),0.0714,0.0037,0.0957,0.0022,24.8558,1.7754
Classical - linear,-0.0045,0.0010,0.0401,0.0010,7.2106,0.0148
Classical - RBF,0.0785,0.0040,0.0974,0.0025,28.3006,2.0667
Classical - poly,0.0204,0.0031,0.0820,0.0010,19.0115,0.3979
Majority-class baseline,NaN,NaN,NaN,NaN,NaN,NaN



Pairwise bootstrap comparisons (balanced accuracy):


,mean_diff,ci_low,ci_high,excludes_zero
comparison,,,,
ZZ - Z,0.0036,-0.0173,0.0252,False
ZZ - Classical RBF,-0.0007,-0.0231,0.0213,False
ZZ - Classical linear,0.0694,0.0469,0.0926,True
ZZ - Classical poly,-0.0153,-0.0385,0.0080,False
Z - Classical RBF,-0.0043,-0.0235,0.0138,False



Most frequently selected quantum hyperparameters:
ZZ - bandwidth: {0.05: 41, 0.1: 40, 0.2: 16, 0.5: 3}  reps: {1: 66, 2: 34}
Z  - bandwidth: {0.2: 54, 0.5: 28, 0.1: 15, 0.05: 3}  reps: {1: 56, 2: 44}


## Group-aware re-run (same models, `StratifiedGroupKFold`)

Re-runs ZZFeatureMap, ZFeatureMap, and the three classical kernels with `groups=groups` so no peptide group (overlapping/near-duplicate 9-mers) spans train and test. Same hyperparameter grids and same number of outer/inner folds as the random-split runs above, so the two can be compared directly, including the entanglement comparison (ZZ vs Z) under group-aware splitting.

⚠️ **Timing**: comparable to the random-split runs above (~15 min for ZZFeatureMap, seconds to ~1 min per classical kernel).

In [ ]:
def repeated_stratified_group_kfold(y, groups, n_splits=5, n_repeats=20, random_state=RANDOM_SEED):
    """Mismo rol que RepeatedStratifiedKFold, pero los grupos nunca se dividen
    entre train/test. StratifiedGroupKFold no es repetible de forma nativa,
    asi que esto la corre n_repeats veces con distinto random_state/shuffle."""
    splits = []
    for r in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state + r)
        for tr_idx, te_idx in sgkf.split(np.zeros(len(y)), y, groups):
            splits.append((tr_idx, te_idx))
    return splits

# sanity check -- confirma que ningun grupo se filtra entre train/test
_check_splits = repeated_stratified_group_kfold(y, groups, n_splits=5, n_repeats=3)
_leak = any(set(groups[tr]) & set(groups[te]) for tr, te in _check_splits)
print("Any group leakage across train/test in sanity check:", _leak, "(must be False)")

In [ ]:
t0 = time.time()
res_zz_grp, preds_zz_grp = nested_cv_quantum(X, y, PARAM_GRID_QUANTUM, kind='zz',
                                              n_splits_outer=N_SPLITS_OUTER, n_repeats_outer=N_REPEATS_OUTER,
                                              n_splits_inner=N_SPLITS_INNER, groups=groups, verbose=True)
print(f'ZZFeatureMap (group-aware): {time.time()-t0:.1f}s total')
res_zz_grp.to_csv('results_quantum_zz_grouped.csv', index=False)
json.dump(preds_zz_grp, open('preds_zz_grouped.json', 'w'))

t0 = time.time()
res_z_grp, preds_z_grp = nested_cv_quantum(X, y, PARAM_GRID_QUANTUM, kind='z',
                                            n_splits_outer=N_SPLITS_OUTER, n_repeats_outer=N_REPEATS_OUTER,
                                            n_splits_inner=N_SPLITS_INNER, groups=groups, verbose=True)
print(f'ZFeatureMap (group-aware): {time.time()-t0:.1f}s total')
res_z_grp.to_csv('results_quantum_z_grouped.csv', index=False)
json.dump(preds_z_grp, open('preds_z_grouped.json', 'w'))

classical_results_grp, classical_preds_grp = {}, {}
for kname in ['linear', 'rbf', 'poly']:
    t0 = time.time()
    res, preds = nested_cv_classical(X, y, kname, n_splits_outer=N_SPLITS_OUTER,
                                      n_repeats_outer=N_REPEATS_OUTER, n_splits_inner=N_SPLITS_INNER,
                                      groups=groups, verbose=True)
    print(f'Classical {kname} (group-aware): {time.time()-t0:.1f}s total')
    res.to_csv(f'results_classical_{kname}_grouped.csv', index=False)
    json.dump(preds, open(f'preds_classical_{kname}_grouped.json', 'w'))
    classical_results_grp[kname] = res
    classical_preds_grp[kname] = preds

res_majority_grp, preds_majority_grp = majority_baseline(X, y, n_splits_outer=N_SPLITS_OUTER,
                                                          n_repeats_outer=N_REPEATS_OUTER, groups=groups)
res_majority_grp.to_csv('results_majority_grouped.csv', index=False)
json.dump(preds_majority_grp, open('preds_majority_grouped.json', 'w'))

NameError: name 'repeated_stratified_group_kfold' is not defined

## Group-aware metrics + comparison against the random-split results

Same `per_fold_metrics` / bootstrap machinery as above, applied to the group-aware predictions, plus a side-by-side table showing how much each model's balanced accuracy drops (or doesn't) once redundant peptides can no longer appear in both train and test. A large drop here would mean the random-split numbers earlier in this notebook were still optimistic despite fixes 2.2/2.3.

In [ ]:
metrics_grp = {
    'ZZFeatureMap (group-aware)': per_fold_metrics(preds_zz_grp),
    'ZFeatureMap (group-aware)': per_fold_metrics(preds_z_grp),
    'Classical - linear (group-aware)': per_fold_metrics(classical_preds_grp['linear']),
    'Classical - RBF (group-aware)': per_fold_metrics(classical_preds_grp['rbf']),
    'Classical - poly (group-aware)': per_fold_metrics(classical_preds_grp['poly']),
    'Majority-class baseline (group-aware)': per_fold_metrics(preds_majority_grp),
}

summary_grp_rows = []
for name, m in metrics_grp.items():
    row = {'model': name, 'n_outer_folds': len(m['accuracy'])}
    for key in ['accuracy', 'balanced_accuracy', 'mcc', 'sensitivity', 'specificity']:
        vals = m[key]
        row[f'{key}_mean'] = np.nanmean(vals)
        row[f'{key}_se'] = np.nanstd(vals, ddof=1) / np.sqrt(np.sum(~np.isnan(vals)))
    summary_grp_rows.append(row)
summary_grp_df = pd.DataFrame(summary_grp_rows).set_index('model')
summary_grp_df.to_csv('final_summary_metrics_grouped.csv')

print("=" * 110)
print("GROUP-AWARE SUMMARY -- mean +/- SE over outer test folds (StratifiedGroupKFold)")
print("=" * 110)
display(summary_grp_df[['accuracy_mean','accuracy_se','balanced_accuracy_mean','balanced_accuracy_se',
                         'mcc_mean','mcc_se','sensitivity_mean','specificity_mean']])

# side-by-side: random-split vs group-aware balanced accuracy, for the models run both ways
pairs = [
    ('ZZFeatureMap (entangled)', 'ZZFeatureMap (group-aware)'),
    ('ZFeatureMap (no entanglement)', 'ZFeatureMap (group-aware)'),
    ('Classical - linear', 'Classical - linear (group-aware)'),
    ('Classical - RBF', 'Classical - RBF (group-aware)'),
    ('Classical - poly', 'Classical - poly (group-aware)'),
]
compare_rows = []
for random_name, grouped_name in pairs:
    ba_random = metrics[random_name]['balanced_accuracy'].mean()
    ba_grouped = metrics_grp[grouped_name]['balanced_accuracy'].mean()
    compare_rows.append({'model': random_name, 'balanced_accuracy_random_split': ba_random,
                          'balanced_accuracy_group_aware': ba_grouped,
                          'drop': ba_random - ba_grouped})
compare_df = pd.DataFrame(compare_rows).set_index('model')
compare_df.to_csv('random_vs_group_aware_comparison.csv')
print()
print("Random-split vs group-aware balanced accuracy (positive 'drop' = random split was optimistic):")
display(compare_df)

# entanglement check under group-aware splitting: does ZZ still beat Z once redundant peptides can't leak?
diff_grp_entangle = metrics_grp['ZZFeatureMap (group-aware)']['balanced_accuracy'] - metrics_grp['ZFeatureMap (group-aware)']['balanced_accuracy']
mean_e, lo_e, hi_e = bootstrap_paired_ci(diff_grp_entangle)
print()
print(f"Group-aware ZZ - Z (entanglement check): mean={mean_e:+.4f}  95% CI=[{lo_e:+.4f}, {hi_e:+.4f}]  "
      f'{"(excludes zero)" if (lo_e>0 or hi_e<0) else "(includes zero)"}')

GROUP-AWARE SUMMARY -- mean +/- SE over outer test folds (StratifiedGroupKFold)


,accuracy_mean,accuracy_se,balanced_accuracy_mean,balanced_accuracy_se,mcc_mean,mcc_se,sensitivity_mean,specificity_mean
model,,,,,,,,
ZZFeatureMap (group-aware),0.8052,0.0089,0.8091,0.0090,0.6119,0.0176,0.7974,0.8207
ZFeatureMap (group-aware),0.7773,0.0116,0.7854,0.0116,0.5721,0.0220,0.7160,0.8549
Classical - linear (group-aware),0.7768,0.0123,0.7829,0.0128,0.5622,0.0243,0.7235,0.8422
Classical - RBF (group-aware),0.7709,0.0115,0.7778,0.0118,0.5550,0.0226,0.7133,0.8423
Classical - poly (group-aware),0.7643,0.0123,0.7686,0.0122,0.5366,0.0238,0.7374,0.7997
Majority-class baseline (group-aware),0.4437,0.0140,0.5000,0.0000,0.0000,0.0000,0.1900,0.8100



Random-split vs group-aware balanced accuracy (positive 'drop' = random split was optimistic):


,balanced_accuracy_random_split,balanced_accuracy_group_aware,drop
model,,,
ZZFeatureMap (entangled),0.8210,0.8091,0.0119
ZFeatureMap (no entanglement),0.7954,0.7854,0.0099
Classical - linear,0.7854,0.7829,0.0025
Classical - RBF,0.8064,0.7778,0.0286
Classical - poly,0.7821,0.7686,0.0135



Group-aware ZZ - Z (entanglement check): mean=+0.0236  95% CI=[+0.0029, +0.0439]  (excludes zero)


### Reading the group-aware results

- If `drop` in the comparison table above is small (a few percentage points or less), the earlier fixes (2.2 scaler, 2.3 nested CV) were already doing most of the work, and the sequence redundancy in this dataset isn't badly inflating the numbers.
- If `drop` is large, report the group-aware numbers as the primary result in the paper and mention the random-split numbers only as a comparison point showing how much redundancy was inflating them -- this is exactly the situation Steve's note anticipates ("if peptides overlap or are highly similar, cluster them first and use group-aware splits").
- Either way, both tables should go in the paper or its supplement so the reviewer can see the effect directly rather than take a group-aware claim on faith.

### Notes for the paper

- Entanglement (ZZ vs Z) remains the clearest, best-supported result: paired CI excludes zero.
- Quantum vs classical (any of linear/RBF/poly) should be read from the bootstrap table above once run on your machine — do not assume the earlier RBF-only result generalizes to linear/poly without checking.
- KTA/CKA/effective-rank let you sanity-check whether higher alignment corresponds to higher accuracy across kernels — Steve's point in section 2.4 is that this correspondence should not be assumed and must be shown, not asserted.
- Group-aware splitting is still not implemented (see the note at the top of this notebook for the redundancy evidence) — flagged so it doesn't get lost before submission.
- Group-aware splitting IS now implemented as a separate section below (`StratifiedGroupKFold`) -- compare its results against the random-split numbers above before deciding which to report as primary in the paper.

In [ ]:
import subprocess

subprocess.run([
    "ffplay",
    "-nodisp",
    "-autoexit",
    "/home/luispabloelchido/repo/mario.ogg"
])